Let's use the pima indian diabetes dataset to predict whether a person has diabetes or not. We will train a standalone model first, then use bagging ensemble technique to see how it can improve the model's performance.

dataset credit: https://www.kaggle.com/gargmanas/pima-indians-diabetes

In [1]:
import pandas as pd
df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [3]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [4]:
df.Outcome.value_counts()

,count
Outcome,
0,500
1,268


There is a slight imbalance in our dataset but since it is not major, we will not worry about it

###Train test split

In [5]:
X = df.drop("Outcome", axis="columns")
y = df.Outcome

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:3]

array([[ 0.63994726,  0.84832379,  0.14964075,  0.90726993, -0.69289057,
         0.20401277,  0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575,  0.53090156, -0.69289057,
        -0.68442195, -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, -1.28821221, -0.69289057,
        -1.10325546,  0.60439732, -0.10558415]])

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, stratify=y, random_state=10)

In [8]:
X_train.shape

(576, 8)

In [9]:
X_test.shape

(192, 8)

In [10]:
y_train.value_counts()

,count
Outcome,
0,375
1,201


In [11]:
y_test.value_counts()

,count
Outcome,
0,125
1,67


###Train using standalone model

In [12]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
scores = cross_val_score(DecisionTreeClassifier(), X, y, cv=5)
scores

array([0.68181818, 0.64285714, 0.67532468, 0.79738562, 0.7124183 ])

In [13]:
scores.mean()

np.float64(0.7019607843137254)

###Train using bagging

In [16]:
from sklearn.ensemble import BaggingClassifier
bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=0
)
bag_model.fit(X_train, y_train)
bag_model.oob_score_

0.7534722222222222

In [17]:
bag_model.score(X_test, y_test)

0.7760416666666666

In [18]:
bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=0
)
scores = cross_val_score(bag_model, X, y, cv=5)
scores

array([0.75324675, 0.72727273, 0.74675325, 0.82352941, 0.73856209])

In [19]:
scores.mean()

np.float64(0.7578728461081402)

There is some improvement in the test score with bagging classifier compared to a standalone classifier

###Train using random forest

In [20]:
from sklearn.ensemble import RandomForestClassifier
scores = cross_val_score(RandomForestClassifier(n_estimators=50), X, y, cv=5)
scores.mean()

np.float64(0.7578813343519226)

###Another dataset: Heart failure prediction

https://www.kaggle.com/fedesoriano/heart-failure-prediction

###Data loading

In [21]:
df = pd.read_csv('heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [23]:
df.shape

(918, 12)

In [22]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


###Treat outliers

In [24]:
df = df[df['Cholesterol']<=(df['Cholesterol'].mean()+3*df['Cholesterol'].std())]
df.shape

(915, 12)

In [25]:
df = df[df['RestingBP']<=(df['RestingBP'].mean()+3*df['RestingBP'].std())]
df.shape

(908, 12)

In [26]:
df = df[df['RestingBP']>=(df['RestingBP'].mean()-3*df['RestingBP'].std())]
df.shape

(907, 12)

In [27]:
df = df[df['MaxHR']>=(df['MaxHR'].mean()-3*df['MaxHR'].std())]
df.shape

(906, 12)

In [28]:
df = df[df['Oldpeak']<=(df['Oldpeak'].mean()+3*df['Oldpeak'].std())]
df.shape

(900, 12)

In [29]:
df = df[df['Oldpeak']>=(df['Oldpeak'].mean()-3*df['Oldpeak'].std())]
df.shape

(899, 12)

###Label/One hot encoding

In [30]:
df.ChestPainType.unique()

array(['ATA', 'NAP', 'ASY', 'TA'], dtype=object)

In [31]:
df.RestingECG.unique()

array(['Normal', 'ST', 'LVH'], dtype=object)

In [32]:
df.ExerciseAngina.unique()

array(['N', 'Y'], dtype=object)

In [33]:
df.ST_Slope.unique()

array(['Up', 'Flat', 'Down'], dtype=object)

In [34]:
df.replace({
    'RestingECG': {'Normal': 1, 'ST': 2, 'LVH': 3},
    'ExerciseAngina': {'N': 0, 'Y': 1},
    'ST_Slope': {'Up': 3, 'Flat': 2, 'Down': 1}
}, inplace=True)
df.head()

/tmp/ipython-input-470318758.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,1,172,0,0.0,3,0
1,49,F,NAP,160,180,0,1,156,0,1.0,2,1
2,37,M,ATA,130,283,0,2,98,0,0.0,3,0
3,48,F,ASY,138,214,0,1,108,1,1.5,2,1
4,54,M,NAP,150,195,0,1,122,0,0.0,3,0


In [35]:
df = pd.get_dummies(df, drop_first=True)
df.head()

,Age,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA
0,40,140,289,0,1,172,0,0.0,3,0,True,True,False,False
1,49,160,180,0,1,156,0,1.0,2,1,False,False,True,False
2,37,130,283,0,2,98,0,0.0,3,0,True,True,False,False
3,48,138,214,0,1,108,1,1.5,2,1,False,False,False,False
4,54,150,195,0,1,122,0,0.0,3,0,True,False,True,False


In [36]:
X = df.drop('HeartDisease', axis='columns')
y = df.HeartDisease

X.head()

,Age,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA
0,40,140,289,0,1,172,0,0.0,3,True,True,False,False
1,49,160,180,0,1,156,0,1.0,2,False,False,True,False
2,37,130,283,0,2,98,0,0.0,3,True,True,False,False
3,48,138,214,0,1,108,1,1.5,2,False,False,False,False
4,54,150,195,0,1,122,0,0.0,3,True,False,True,False


###Scaling

In [37]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

array([[-1.42815446,  0.46590022,  0.84963584, ...,  2.06332497,
        -0.5349047 , -0.22955001],
       [-0.47585532,  1.63471366, -0.16812204, ..., -0.48465463,
         1.86949191, -0.22955001],
       [-1.7455875 , -0.1185065 ,  0.79361247, ...,  2.06332497,
        -0.5349047 , -0.22955001],
       ...,
       [ 0.3706328 , -0.1185065 , -0.62564622, ..., -0.48465463,
        -0.5349047 , -0.22955001],
       [ 0.3706328 , -0.1185065 ,  0.35476274, ...,  2.06332497,
        -0.5349047 , -0.22955001],
       [-1.63977649,  0.34901888, -0.21480818, ..., -0.48465463,
         1.86949191, -0.22955001]])

In [38]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=20)

In [39]:
X_train.shape

(719, 13)

In [40]:
X_test.shape

(180, 13)

###Train a model using standalone SVM

In [41]:
from sklearn.svm import SVC
scores = cross_val_score(SVC(), X, y, cv=5)
scores.mean()

np.float64(0.6918001241464928)

###Bagging

In [42]:
bag_model = BaggingClassifier(
    estimator=SVC(),
    n_estimators=100,
    max_samples=0.8,
    random_state=0
)
scores = cross_val_score(bag_model, X, y, cv=5)
scores.mean()

np.float64(0.685127250155183)

Using bagging instead of standalone SVM doesn't make much difference in terms of model accuracy.

Bagging is effective when we have a high variance and an unstable model such as decision tree.

Let's explore how bagging changes the performance of a decision tree classifier.

###Train a model using decision tree

In [43]:
scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=5)
scores.mean()

np.float64(0.7151024208566108)

###Bagging

In [44]:
bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=0),
    n_estimators=100,
    max_samples=0.8,
    random_state=0
)
scores = cross_val_score(bag_model, X, y, cv=5)
scores.mean()

np.float64(0.8007945375543141)

With bagging, the score improved from 71.5% to 80.1%

###Train a model using Random forest which uses bagging underneath

In [45]:
scores = cross_val_score(RandomForestClassifier(), X, y, cv=5)
scores.mean()

np.float64(0.8208255741775294)

Random forest gave the best performance with a 82.1% score. It uses bagging where it samples not only data rows but also the columns (features)